In [1]:
import os
import json
import pandas as pd # openpyxl  - need as well for pd.read_excel
import json
from pathlib import Path
import numpy as np
from shapely.geometry import shape, mapping
from shapely.ops import orient

In [15]:
# Load in the GeoJSON objects
AREAS = {
    "maori":("./raw/maori-electorates-2020.json", "MED2020_V1_00")
    , "general":("./raw/general-electorates-2020.json", "GED2020_V1_00")
}
# json.loads(Path(link).read_text("utf-8"))
loaded_maps = [
    (json.loads(Path((AREAS[x][0])).read_text("utf-8")),AREAS[x][1], x) for x in AREAS
]

In [17]:
# Combine the Maori and General electorates.
gen_electorate_names = [x["properties"]["name"]  for x  in loaded_maps[1][0]["features"]]
m_electorate_names = [x["properties"]["name"]  for x  in loaded_maps[0][0]["features"]]
electorate_names = gen_electorate_names + m_electorate_names

In [19]:
# Creates "electorateInfo.json - which is just a file with the different electorates and two empty fields that can manually entered"
pd.DataFrame.from_dict([{"Electorate":x, "Type": "", "Region":""} for x in electorate_names]).to_json("./clean/electorateInfo.json", orient="records")

In [5]:
# This retrieves the party vote / electoral vote numbers for each electorate

# Party Vote
df = pd.read_csv("./raw/votes-for-registered-parties-by-electorate.csv", skiprows=1, encoding="UTF-8")
# Electorate Vote
elec_df = pd.read_csv("./raw/percentage-candidate-votes-for-registered-parties.csv", skiprows=1, encoding="UTF-8")

In [ ]:
# Creates "party_vote.json" - stores the count of party votes for each electorate for both percent and total

# Party renaming code table
party_code_table = { 
    "ACT New Zealand": "ACT"
    , "New Zealand First Party": "NZ First"
    , "National Party": "National"
    , "Labour Party": "Labour"
    , "Green Party" : "Greens"
    , "Te Pāti Māori" : "Māori"
    , "The Opportunities Party (TOP)" : "TOP"
    , "Informal Party Votes": "Informal"
    , "Other": "Other"
    , "Total Party Votes Counted": "Total"
}

# Data transformation
ignore_cols = ["Electorate", "Total Valid Party Votes"]
ignore_rows = ['General Electorate Totals', 'Māori Electorate Totals', 'Combined Totals']
other_parties = [x for x in df.columns if x not in party_code_table.keys() and x not in ignore_cols]
party_vote_df = df.assign(Other = lambda x: x[other_parties]\
          .sum(axis=1))\
           [["Electorate"] + list(party_code_table.keys())]\
           .rename(columns=party_code_table)\
           .query(f'Electorate not in {ignore_rows}')
          

# DATA VALIDATION

test_rows = len(party_vote_df.assign(Test_Total=lambda x: x.drop(columns=["Electorate", "Total"]).sum(axis=1))\
    .query("Test_Total != Total"))


# Check rows missing
if (test_rows != 0):
    print("WARNING: Total did not reconcile")

# Check electorate names match
missing_electorates = list(set(party_vote_df["Electorate"].unique()) - set(electorate_names))
# Check the count is 72
electorates_count = len(set(party_vote_df["Electorate"].unique()))

if electorates_count != 72:
    print(f"WARNING: Only {electorates_count} of 72")

if len(missing_electorates) > 0:
    print(f"WARNING: Missing electorates {missing_electorates}")


# END OF DATA VALIDATION

# Calculates percentage from the number of votes
percent_vote_df = party_vote_df.copy()
for col in percent_vote_df.drop(columns=["Electorate", "Total"]).columns:
    percent_vote_df[col] = round((percent_vote_df[col] * 100) / percent_vote_df["Total"],2)

percent_vote_df = percent_vote_df.drop(columns=["Total"])
    
party_vote_df["Stat"] = "Total"
percent_vote_df["Stat"] = "Percent"

# Data transformation
party_df = pd.concat(
    [party_vote_df.melt(id_vars=["Electorate", "Stat"], var_name="Party", value_name="Votes")
    , percent_vote_df.melt(id_vars=["Electorate", "Stat"], var_name="Party", value_name="Votes")]
).reset_index(drop=True)

party_df.to_json("./clean/party_vote.json", orient="records")

,Electorate,Stat,Party,Votes
0,Auckland Central,Total,ACT,3301.00
1,Banks Peninsula,Total,ACT,3919.00
2,Bay of Plenty,Total,ACT,5212.00
3,Botany,Total,ACT,2788.00
4,Christchurch Central,Total,ACT,2646.00
...,...,...,...,...
1363,Tāmaki Makaurau,Percent,Other,4.15
1364,Te Tai Hauāuru,Percent,Other,4.16
1365,Te Tai Tokerau,Percent,Other,4.91
1366,Te Tai Tonga,Percent,Other,5.07


In [29]:

# Creates "electorate_vote.json" - stores the count of candidate votes for each electorate for both percent and total

# Same logic as above - data transformations
electorate = "Electorate"

parties = ["ACT", "Greens", "Labour", "National", "NZ First", "Māori", "Other"]

elec_df_whole = elec_df\
    .rename(columns = party_code_table)\
    .rename(columns = {"Unnamed: 0": "Electorate"})[1:74]\
    .query("Electorate != 'General Electorate Totals'")\
    [[electorate] + parties]\
    .astype({x: int for x in parties})\
    .assign(Total= lambda x: x[parties].sum(axis=1), Stat="Total")

elec_df_pct = elec_df_whole\
    .assign(**{x: (lambda y, x = x: round((y[x] / y["Total"]) * 100, 2)) for x in parties})\
    .drop(columns=["Total"])\
    .assign(Stat="Percent")


elec_df_final = pd.concat(
    [elec_df_whole.melt(id_vars=["Electorate", "Stat"], var_name="Party", value_name="Votes")
    , elec_df_pct.melt(id_vars=["Electorate", "Stat"], var_name="Party", value_name="Votes")]
).query("Votes != 0").reset_index(drop=True)

elec_df_final.to_json("./clean/electorate_vote.json", orient="records")

In [8]:
# Function to help automate transformation demographics from the census excel spreadsheets
# Only works for binned data (and not ethniciity)

def clean_sheet(sheet_name, remove_cols = [], merge_cols=[], merge_cols_names=[]):

    # Different sheets have Total Stated and Total
    general_sheet_df = pd.read_excel("./raw/general_electorate_census2023_data.xlsx", sheet_name=sheet_name, skiprows=2)\
        .assign(Total = lambda x: x["Total Stated"] if "Total Stated" in x.columns else x["Total"])\
        .drop(columns=["Total Stated", ""] + remove_cols, axis=1, errors="ignore")\
        .rename(columns={"Unnamed: 0": "Electorate"})\
        .query("Electorate not in ('Area outside general electoral district', 'Total')")\
        
    maori_sheet_df = pd.read_excel("./raw/māori_electorate_census2023_data.xlsx", sheet_name=sheet_name, skiprows=2)\
        .assign(Total = lambda x: x["Total Stated"] if "Total Stated" in x.columns else x["Total"])\
        .drop(columns=["Total Stated"] + remove_cols, axis=1, errors="ignore")\
        .rename(columns={"Unnamed: 0": "Electorate"})\
        .query("Electorate not in ('Area Outside Māori Electoral District', 'Total')")\
    
    sheet_df = pd.concat([general_sheet_df, maori_sheet_df])

    # Check electorate names match
    missing_electorates = list(set(sheet_df["Electorate"].unique()) - set(electorate_names))
    # Check the count is 72
    electorates_count = len(set(sheet_df["Electorate"].unique()))

    if electorates_count != 72:
        print(f"WARNING: Only {electorates_count} of 72")

    if len(missing_electorates) > 0:
        print(f"WARNING: Missing electorates {missing_electorates}")


    for merge, name in zip(merge_cols, merge_cols_names):
        sheet_df = sheet_df.assign(_temp_col=lambda x: x[list(merge)].sum(axis=1))\
            .rename(columns={"_temp_col": name})\
            .drop(list(merge), axis=1)

    pct_df = sheet_df.copy()
    for col in pct_df.drop(columns=["Electorate", "Total"]).columns:
        pct_df[col] = round((pct_df[col] * 100) / pct_df["Total"],2)
    pct_df = pct_df.drop(columns=["Total"])

    sheet_df["Stat"] = "Total"
    pct_df["Stat"] = "Percent"

    final_df = pd.concat([
        sheet_df.melt(id_vars=["Electorate", "Stat"], var_name="Category", value_name="Value")
        , pct_df.melt(id_vars=["Electorate", "Stat"], var_name="Category", value_name="Value")
    ]).reset_index(drop=True)

    final_df["Census"] = sheet_name

    return final_df.copy()

In [9]:
# Custom logic to calculate the previous age

general_sheet_df = pd.read_excel("./raw/general_electorate_census2023_data.xlsx", sheet_name="Age", skiprows=2)\
        .assign(Total = lambda x: x["Total Stated"] if "Total Stated" in x.columns else x["Total"])\
        .drop(columns=["Total Stated", "", "Total"] , axis=1, errors="ignore")\
        .rename(columns={"Unnamed: 0": "Electorate"})\
        .query("Electorate not in ('Area outside general electoral district', 'Total')")

maori_sheet_df = pd.read_excel("./raw/māori_electorate_census2023_data.xlsx", sheet_name="Age", skiprows=2)\
        .assign(Total = lambda x: x["Total Stated"] if "Total Stated" in x.columns else x["Total"])\
        .drop(columns=["Total Stated", "", "Total"], axis=1, errors="ignore")\
        .rename(columns={"Unnamed: 0": "Electorate"})\
        .query("Electorate not in ('Area Outside Māori Electoral District', 'Total')")

sheet_df = pd.concat([general_sheet_df, maori_sheet_df]).reset_index(drop=True)

median_ages = []

for electorate in sheet_df["Electorate"]:

    
    median_df = sheet_df[sheet_df.Electorate == electorate].copy()\
        .transpose()[1:]\
        .reset_index()
    
    median_df.columns = ["Bin", "Count"]

    median_df = median_df\
        .assign(Lower_Bound = lambda x: x["Bin"].str.split(r"-| ").str[0].astype(int))\
        .assign(Cumsum= lambda x: x["Count"].cumsum())
    
    median_loc = median_df["Count"].sum() / 2

    index = median_df["Cumsum"].searchsorted(median_loc)

    prevrow = median_df.iloc[index-1]
    row = median_df.iloc[index]

    median = round(row.Lower_Bound + ((median_loc - prevrow.Cumsum) / row.Count) * 5,2)
    
    median_ages.append({"Electorate":electorate, "Census": "Median age", "Category": "Median age", "Value": median})


median_df = pd.DataFrame.from_dict(median_ages)

In [10]:
# Custom logic to get the median/mean household incomes

general_sheet_df = pd.read_excel("./raw/general_electorate_census2023_data.xlsx", sheet_name="Household income", skiprows=1)\
        .rename(columns={"General Electoral District": "Electorate"})\
        .query("Electorate not in ('Area outside general electoral district', 'Total')")

maori_sheet_df = pd.read_excel("./raw/māori_electorate_census2023_data.xlsx", sheet_name="Household income", skiprows=2)\
    .rename(columns={"Unnamed: 0": "Electorate"})\
    .query("Electorate not in ('Area Outside Māori Electoral District', 'Total')")

income_df = pd.concat([general_sheet_df, maori_sheet_df])\
    .reset_index(drop=True).melt(id_vars="Electorate",var_name="Census", value_name="Value").assign(Category = lambda x: x["Census"])

In [21]:
# Custom logic to get ethnicity - also worth noting that ethnicity will not add up to 100% as people can have more than 1
# No ethnicity estimates for Maori as they are 100% Maori

general_sheet_df = pd.read_excel("./raw/general_electorate_census2023_data.xlsx", sheet_name="Ethnicity L1", skiprows=2)\
    .assign(Total = lambda x: x["Total Stated"] if "Total Stated" in x.columns else x["Total"])\
    .drop(columns=["Total stated", ""] , axis=1, errors="ignore")\
    .rename(columns={"Unnamed: 0": "Electorate"})\
    .query("Electorate not in ('Area outside general electoral district', 'Total')")


normal_df = general_sheet_df.copy()\
    .melt(id_vars=["Electorate", "Total"], var_name="Category", value_name="Value")\
    .assign(Stat="Total")\
    .drop(columns=["Total"])\
    .reset_index(drop=True)

pct_df = general_sheet_df.copy()\
    .melt(id_vars=["Electorate", "Total"], var_name="Category", value_name="Value")\
    .assign(Value=lambda x:round(x["Value"] / x["Total"] * 100, 2), Stat="Percent")\
    .drop(columns=["Total"])\
    .reset_index(drop=True)


# Creating a fake maori DF

electorate_pops = {
    "Hauraki-Waikato": 135930 
    , "Ikaroa-Rāwhiti": 122286
    , "Tāmaki Makaurau": 134205
    , "Te Tai Hauāuru": 140685
    , "Te Tai Tokerau": 139662
    , "Te Tai Tonga": 182040
    , "Waiariki": 123432
}

non_maori_ethnicities = [x for x in normal_df.Category.unique() if x != "Māori"]

maori_df = pd.DataFrame.from_records(
    [{"Electorate": name, "Category": "Māori", "Value": 100.0, "Stat": "Percent"} for name in m_electorate_names] 
    + [{"Electorate": name, "Category": eth_name, "Value": 0.0, "Stat": "Percent"} for eth_name in non_maori_ethnicities for name in m_electorate_names] 
    + [{"Electorate": name, "Category": "Māori", "Value": electorate_pops[name], "Stat": "Total"} for name in m_electorate_names] 
    + [{"Electorate": name, "Category": eth_name, "Value": 0.0, "Stat": "Total"} for eth_name in non_maori_ethnicities for name in m_electorate_names] 
)

ethnicity_df = pd.concat([normal_df, pct_df, maori_df]).copy().assign(Census="Ethnicity").reset_index(drop=True)

#ethnicity_df.query("Electorate=='Auckland Central'")
ethnicity_df.Category = ethnicity_df.Category.apply(lambda x: x + " Ethnicity" if x == "Māori" else x)
ethnicity_df.Category = ethnicity_df.Category.apply(lambda x: "MENA" if x == "Middle Eastern/Latin American/African" else x)

ethnicity_df.query("Electorate == 'Waiariki'")

,Electorate,Category,Value,Stat,Census
786,Waiariki,Māori Ethnicity,100.0,Percent,Ethnicity
793,Waiariki,European,0.0,Percent,Ethnicity
800,Waiariki,Pacific Peoples,0.0,Percent,Ethnicity
807,Waiariki,Asian,0.0,Percent,Ethnicity
814,Waiariki,MENA,0.0,Percent,Ethnicity
821,Waiariki,Other Ethnicity,0.0,Percent,Ethnicity
828,Waiariki,Māori Ethnicity,123432.0,Total,Ethnicity
835,Waiariki,European,0.0,Total,Ethnicity
842,Waiariki,Pacific Peoples,0.0,Total,Ethnicity
849,Waiariki,Asian,0.0,Total,Ethnicity


In [104]:

configs = [
    {"sheet_name": "Age"}
    #, {"sheet_name": "Ethnicity L1"} - Got to handle separately due to Maori
    , {
        "sheet_name": "Sexual identity"
        , "merge_cols": [("Bisexual", "Homosexual", "Sexual identity not elsewhere classified")]
        , "merge_cols_names": ["LGBT"]
        , "remove_cols":["Prefer not to say", "Not Elsewhere Included"]
    },
    {
        "sheet_name": "Labour force status"
        , "remove_cols":["Work and Labour Force Status Unidentifiable"]
    },
    {
        "sheet_name": "Highest qualification"
        , "merge_cols": [
            ("Level 1 certificate", "Level 2 certificate", "Level 3 certificate", "Overseas secondary school qualification")
            , ("Level 4 certificate", "Level 5 diploma", "Level 6 diploma")
            , [("Bachelor degree and Level 7 qualification")]
            , ("Post-graduate and honours degrees", "Masters degree", "Doctorate degree")
        ]
        , "merge_cols_names": ["High School", "Some tertiary", "Bachelors", "Post-grad"]
        , "remove_cols":["Not elsewhere included"]
    },
    {
        "sheet_name": "Religious affiliation"
        , "merge_cols" : [
            ("Buddhism", "Judaism", "Māori Religions, Beliefs and Philosophies", "Other Religions, Beliefs and Philosophies", "Spiritualism and New Age Religions")
        ]
        , "merge_cols_names": ["Other Religion"]
        , "remove_cols":["Residual Categories", "Total stated"]
    },
    {
        "sheet_name": "Occupation"
        , "merge_cols" : [
            ("Managers", "Professionals", "Clerical and Administrative Workers")
            , ("Technicians and Trades Workers", "Labourers", "Machinery Operators and Drivers")
            , ( "Community and Personal Service Workers", "Sales Workers")
        ]
        , "merge_cols_names": ["White Collar", "Blue Collar", "Services"]
        , "remove_cols": ["Residual Categories (Operational Codes only)", "Total stated"]
    }
]

final_df = pd.concat([clean_sheet(**config) for config in configs] + [median_df, income_df, ethnicity_df]).reset_index(drop=True)
    
final_df.to_json("./clean/census.json", orient="records")

In [15]:
for loaded_map, area_col, map_name in loaded_maps:

    count_area = 0

    for area in loaded_map["features"]:


        geom = shape(area['geometry'])
        fixed_geom = orient(geom, sign=-1.0) 
        area['geometry'] = mapping(fixed_geom)

In [16]:
TARGET_PATH = "./clean/"

for loaded_map, _, name in loaded_maps:
    path = Path(TARGET_PATH + name + ".json")


    with open(path, "w", encoding="utf-8") as f:
        json.dump(loaded_map,f)